# Week 1: Explore support ticket data

**Question:** Can short ticket messages be routed to account, billing, or technical support? The label is the destination queue. Our first goal is to inspect data quality and class balance before modeling.


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from ticket_classifier import DATA, load_data

data = load_data(DATA)
data.head()


## Check missing values, duplicates, and class balance

`load_data` strips whitespace, removes empty texts and unsupported labels, and drops exact duplicate text. Inspect the raw file too if you change the dataset.


In [ ]:
raw = pd.read_csv(DATA)
print("Raw shape:", raw.shape)
print("Missing values:\n", raw.isna().sum())
print("Duplicate texts:", raw.duplicated("text").sum())
print("Clean shape:", data.shape)
print("Class counts:\n", data["label"].value_counts())


In [ ]:
data["word_count"] = data["text"].str.split().str.len()
print(data.groupby("label")["word_count"].describe().round(1))
print("Word-count percentiles:", np.percentile(data["word_count"], [25, 50, 75]))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
data["label"].value_counts().sort_index().plot.bar(ax=axes[0], title="Tickets per queue")
sns.histplot(data=data, x="word_count", hue="label", binwidth=2, multiple="layer", ax=axes[1])
axes[1].set_title("Ticket length by queue")
axes[1].set_xlabel("Words")
plt.tight_layout()


## Inspect examples and state risks

Look for ambiguous wording, label mistakes, and repeated templates. This teaching dataset is hand-written and balanced, so performance on it will not estimate performance on live traffic.


In [ ]:
for label in sorted(data["label"].unique()):
    print(f"\n{label.upper()}")
    for text in data.loc[data["label"] == label, "text"].sample(3, random_state=42):
        print("-", text)


## Next step

Run `python ticket_classifier.py train` in a terminal. Compare macro F1 and per-class recall with the majority-class baseline. Open `artifacts/test_predictions.csv` to inspect mistakes.
